# Final test — FLOP-matched bake-off (LR-tuned)

Pruned, T4-friendly study: **fewer configs**, **smaller model**, **more tokens per trial** so you leave the high-loss transient regime.

**Question:** Under equal FLOPs, does SOAP-early + lighter/different late beat full SOAP / AdamW / Muon?

| Piece | Spec |
|-------|------|
| Data | FineWeb-Edu `sample-10BT`, ~30M-token shard |
| Model | nanoGPT-style ~`6L / 4H / 256d` |
| Budget | AdamW FLOPs/step × `REF_STEPS` (Colab default targets ~tens of M tokens seen) |
| Depths | ~25% / 50% / 75% bottom |
| Full | AdamW, SOAP, Muon |
| Splits | soap|*, muon|*, adamw|{soap,sgd,muon} × 3 depths |

| Section | Purpose |
|---------|---------|
| **A** | Install + config |
| **B** | Library |
| **C** | FLOP trainer |
| **D** | Data |
| **E** | Budget calibration |
| **F** | Runs |
| **G** | Plots + summary |

`MODE="vet"` = tiny local smoke. `MODE="colab"` = real FineWeb bake-off.

**LRs:** locked from thorough `study_lr_tune` (AdamW / SOAP / Muon interior winners).
**Save:** final cell zips `plots/study` → Google Drive.

**Fairness (this run):** Same model/data/seed; equal FLOP budget incl. optimizer cost; full-opt LRs from a thorough interior grid (`late_mean`); splits inherit those LRs. Caveats: one seed; tune horizon shorter than bake-off; SGD/Shampoo not retuned.


## A1. Install

Run once. NumPy import error → **Runtime → Restart session**, then Run All.


In [ ]:
%pip install -q datasets tiktoken tqdm heavyball matplotlib


## A2. Config

**Edit this cell only**, then Run All.


In [ ]:
# ===================== EDIT ME =====================
MODE = "colab"                 # "vet" | "colab"
FORCE_SYNTHETIC = None
SEED = 42
RESET_DATA = False             # True only to wipe shards; otherwise 30M shard is reused
RESUME = False                 # False after LR retune (or delete bakeoff_checkpoints.json)

# FLOP budget ≈ AdamW FLOPs/step × REF_STEPS
# Colab: ~5000 steps × 16 × 256 ≈ 20M tokens seen for AdamW (order-of-magnitude)
REF_OPT = "adamw"
REF_STEPS = 5000
FLOP_BUDGET = None

# Depth cuts as fraction of blocks in BOTTOM group
DEPTH_FRACS = [0.25, 0.50, 0.75]

# From thorough LR tune (study_lr_tune_READY, late_mean, interior winners)
LOCKED_LR = {
    "adamw":   0.000621179,
    "sgd":     1e-1,          # not retuned; only used in soap|sgd split
    "muon":    0.00582957,
    "soap":    0.00257309,
    "shampoo": 1e-2,          # unused in default bake-off
}

FULL_OPTS = ["adamw", "soap", "muon"]
SPLIT_COMBOS = [
    ("soap", "muon"),
    ("soap", "adamw"),
    ("soap", "sgd"),
    ("soap", "soap"),
    ("muon", "adamw"),
    ("muon", "soap"),
    ("muon", "muon"),
    ("adamw", "soap"),
    ("adamw", "sgd"),
    ("adamw", "muon"),
]

CALIB_WARMUP = 1
CALIB_MEASURE = 3
# ===================================================

import math
import shutil
import torch
from pathlib import Path


def frac_to_split(frac: float, n_layer: int) -> int:
    k = int(math.floor(frac * n_layer + 0.5))
    return max(1, min(n_layer - 1, k))


def unique_depth_cuts(fracs, n_layer):
    cuts, skipped, seen = [], [], set()
    for f in fracs:
        s = frac_to_split(f, n_layer)
        if s in seen:
            skipped.append((f, s))
            continue
        seen.add(s)
        cuts.append({"frac": f, "split_at": s})
    return cuts, skipped


if MODE == "colab":
    if FORCE_SYNTHETIC is None:
        FORCE_SYNTHETIC = False
    # Smaller nanoGPT-style GPT (embed still dominates GPT-2 vocab)
    N_LAYER, N_HEAD, N_EMBD, BLOCK, BS = 6, 4, 256, 256, 16
    TOKEN_BUDGET, VAL_TOKENS = 30_000_000, 50_000
    CFG = dict(
        token_budget=TOKEN_BUDGET, val_tokens=VAL_TOKENS,
        n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, block_size=BLOCK,
        batch_size=BS, steps=REF_STEPS,
        eval_every=max(100, REF_STEPS // 25),
        split_at=2,
        combo_search="locked",
        force_synthetic=FORCE_SYNTHETIC,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    if CFG["device"] != "cuda":
        print("WARNING: no CUDA — Colab bake-off on CPU is impractical.")
else:
    if FORCE_SYNTHETIC is None:
        FORCE_SYNTHETIC = True
    REF_STEPS = 40
    FULL_OPTS = ["adamw", "soap", "muon"]
    SPLIT_COMBOS = [
        ("soap", "muon"),
        ("soap", "adamw"),
        ("soap", "soap"),
        ("muon", "adamw"),
    ]
    DEPTH_FRACS = [0.25, 0.50, 0.75]
    N_LAYER, N_HEAD, N_EMBD, BLOCK, BS = 4, 2, 64, 64, 8
    CFG = dict(
        token_budget=20_000, val_tokens=2_000,
        n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, block_size=BLOCK,
        batch_size=BS, steps=REF_STEPS,
        eval_every=10,
        split_at=2,
        combo_search="locked",
        force_synthetic=True,
        device="cpu",
    )

DEPTH_CUTS, DEPTH_SKIPPED = unique_depth_cuts(DEPTH_FRACS, CFG["n_layer"])

# Approx unique params (tied wte/lm_head counted once)
_vocab = 50257
_block = 12 * CFG["n_embd"] ** 2  # rough transformer block
_approx = _vocab * CFG["n_embd"] + CFG["n_layer"] * _block + CFG["block_size"] * CFG["n_embd"]
_tokens_adamw = CFG["batch_size"] * CFG["block_size"] * REF_STEPS

assert set(FULL_OPTS) <= set(LOCKED_LR)
assert all(b in LOCKED_LR and t in LOCKED_LR for b, t in SPLIT_COMBOS)

if RESET_DATA:
    d = Path("./data/fineweb_edu")
    if d.exists():
        shutil.rmtree(d)
        print("removed", d)

n_split = len(DEPTH_CUTS) * len(SPLIT_COMBOS)
print("MODE:", MODE)
print("device:", CFG["device"])
print(f"model: ~{CFG['n_layer']}L / {CFG['n_head']}H / {CFG['n_embd']}d / T={CFG['block_size']}")
print(f"approx params (tied embed): {_approx/1e6:.1f}M")
print(f"token shard: {CFG['token_budget']:,} (val {CFG['val_tokens']:,})")
print(f"REF_STEPS={REF_STEPS} → ~{_tokens_adamw/1e6:.1f}M tokens seen for AdamW (order-of-mag)")
print("REF_OPT:", REF_OPT, " FLOP_BUDGET override:", FLOP_BUDGET)
print("\nDepth cuts:")
for c in DEPTH_CUTS:
    print(f"  {c['frac']:.0%} → split_at={c['split_at']}  "
          f"(bottom={c['split_at']}, top={CFG['n_layer']-c['split_at']})")
if DEPTH_SKIPPED:
    print("Deduped:", DEPTH_SKIPPED)
print(f"\nFULL_OPTS ({len(FULL_OPTS)}):", FULL_OPTS)
print(f"SPLIT_COMBOS ({len(SPLIT_COMBOS)}):", SPLIT_COMBOS)
print(f"Estimated trials: {len(FULL_OPTS)} full + {n_split} split = {len(FULL_OPTS)+n_split}")
print("config:")
for k, v in CFG.items():
    print(f"  {k}: {v}")


## B. Library


In [ ]:
from __future__ import annotations
import math, time, json
from pathlib import Path
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW, SGD, Optimizer

DATA_DIR = Path("./data/fineweb_edu")
OUT_DIR = Path("./plots/study")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- GPT --------------------
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 64
    n_layer: int = 4
    n_head: int = 2
    n_embd: int = 64
    dropout: float = 0.0
    bias: bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head, self.n_embd = cfg.n_head, cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.ln2, self.mlp = nn.LayerNorm(cfg.n_embd), MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight
        self.apply(self._init)
        for pn, p in self.named_parameters():
            if pn.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        for blk in self.blocks:
            x = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def forward_with_block_outs(self, idx):
        """Return final logits and list of post-block hidden states (for spectra)."""
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        outs = []
        for blk in self.blocks:
            x = blk(x)
            outs.append(x)
        logits = self.lm_head(self.ln_f(x))
        return logits, outs

# -------------------- Optimizers --------------------
class CompositeOptimizer:
    def __init__(self, optimizers, name="Composite"):
        self.optimizers, self.name = list(optimizers), name
    def zero_grad(self, set_to_none=True):
        for o in self.optimizers: o.zero_grad(set_to_none=set_to_none)
    def step(self, closure=None):
        for o in self.optimizers: o.step()
    def __repr__(self):
        return f"{self.name}(" + ", ".join(type(o).__name__ for o in self.optimizers) + ")"

def unique_params(model):
    seen, out = set(), []
    for p in model.parameters():
        if p.requires_grad and id(p) not in seen:
            seen.add(id(p)); out.append(p)
    return out

def make_optimizer(name, params, lr):
    name, params = name.lower(), list(params)
    if name == "adamw":
        return AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    if name == "sgd":
        return SGD(params, lr=lr, momentum=0.9)
    if name == "muon":
        m2 = [p for p in params if p.ndim >= 2]
        m1 = [p for p in params if p.ndim < 2]
        opts = []
        if m2: opts.append(torch.optim.Muon(m2, lr=lr, weight_decay=0.1, momentum=0.95))
        if m1: opts.append(AdamW(m1, lr=min(lr, 3e-4) if lr > 1e-3 else lr, betas=(0.9, 0.95), weight_decay=0.1))
        return opts[0] if len(opts) == 1 else CompositeOptimizer(opts, "Muon+AdamW")
    if name in ("soap", "shampoo"):
        import heavyball
        kw = dict(lr=lr, weight_decay=0.01, compile_step=False, betas=(0.9, 0.95))
        return heavyball.SOAP(params, **kw) if name == "soap" else heavyball.KLShampoo(params, **kw)
    raise ValueError(name)

def split_param_groups(model, split_at):
    n = model.cfg.n_layer
    assert 0 < split_at < n
    bottom, top, seen = [], [], set()
    def add(g, ps):
        for p in ps:
            if id(p) not in seen:
                seen.add(id(p)); g.append(p)
    add(bottom, model.wte.parameters()); add(bottom, model.wpe.parameters())
    for b in model.blocks[:split_at]: add(bottom, b.parameters())
    for b in model.blocks[split_at:]: add(top, b.parameters())
    add(top, model.ln_f.parameters()); add(top, model.lm_head.parameters())
    return bottom, top

def build_full_optimizer(model, name, lr):
    return make_optimizer(name, unique_params(model), lr)

def build_split_optimizers(model, split_at, bottom_opt, top_opt, bottom_lr, top_lr):
    b, t = split_param_groups(model, split_at)
    return make_optimizer(bottom_opt, b, bottom_lr), make_optimizer(top_opt, t, top_lr)

def all_split_combos():
    return [(a, b) for a in OPTS for b in OPTS]

# -------------------- Data --------------------
def ensure_data(cfg):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    tr_path, va_path = DATA_DIR / "train.bin", DATA_DIR / "val.bin"

    def write(tokens, val_n):
        tokens[:val_n].tofile(va_path)
        tokens[val_n:].tofile(tr_path)
        print(f"[data] train={tr_path} ({len(tokens)-val_n:,})  val={va_path} ({val_n:,})")

    def shards_ok():
        if not (tr_path.exists() and va_path.exists()):
            return False
        need = cfg["token_budget"] - cfg["val_tokens"]
        tr = np.memmap(tr_path, dtype=np.uint16, mode="r")
        va = np.memmap(va_path, dtype=np.uint16, mode="r")
        ok = len(tr) >= need and len(va) >= min(cfg["val_tokens"], need // 10)
        del tr, va
        return ok

    if cfg["force_synthetic"]:
        import tiktoken
        enc = tiktoken.get_encoding("gpt2")
        eot = enc._special_tokens["<|endoftext|>"]
        text = ("Transformers predict the next token. Early layers mix local syntax; "
                "deeper layers carry longer-range semantics. Optimizers shape updates.")
        ids = []
        while len(ids) < cfg["token_budget"]:
            ids.extend(enc.encode_ordinary(text)); ids.append(eot)
        write(np.array(ids[:cfg["token_budget"]], dtype=np.uint16), cfg["val_tokens"])
        print("[data] SYNTHETIC (vet only — not for the write-up)")
    elif shards_ok():
        print(f"[data] Reusing existing shards in {DATA_DIR} "
              f"(train>={cfg['token_budget']-cfg['val_tokens']:,})")
    else:
        import tiktoken
        from datasets import load_dataset
        from tqdm.auto import tqdm
        enc = tiktoken.get_encoding("gpt2")
        eot = enc._special_tokens["<|endoftext|>"]
        print(f"[data] FineWeb-Edu → {cfg['token_budget']:,} tokens...")
        ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
        ds = ds.shuffle(seed=SEED, buffer_size=10_000)
        tokens = np.empty(cfg["token_budget"], dtype=np.uint16)
        n = 0
        pbar = tqdm(total=cfg["token_budget"], unit="tok")
        for row in ds:
            text = row.get("text") or ""
            if not text.strip(): continue
            for tok in enc.encode_ordinary(text) + [eot]:
                tokens[n] = tok; n += 1; pbar.update(1)
                if n >= cfg["token_budget"]: break
            if n >= cfg["token_budget"]: break
        pbar.close()
        assert n == cfg["token_budget"]
        write(tokens, cfg["val_tokens"])

    train = np.asarray(np.memmap(tr_path, dtype=np.uint16, mode="r"))
    val = np.asarray(np.memmap(va_path, dtype=np.uint16, mode="r"))
    max_tr = cfg["token_budget"] - cfg["val_tokens"]
    return train[:max_tr], val[:cfg["val_tokens"]]

# -------------------- Train --------------------
def get_batch(data, bs, T, device, g):
    ix = torch.randint(0, len(data) - T - 1, (bs,), generator=g)
    x = torch.stack([torch.from_numpy(data[i:i+T].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+T].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def eval_val(model, val, bs, T, device, iters=32):
    model.eval()
    g = torch.Generator().manual_seed(0)
    losses = []
    for _ in range(iters):
        xb, yb = get_batch(val, bs, T, device, g)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def make_model(cfg):
    return GPT(GPTConfig(
        block_size=cfg["block_size"], n_layer=cfg["n_layer"],
        n_head=cfg["n_head"], n_embd=cfg["n_embd"],
    )).to(cfg["device"])

def train_full(cfg, train, val, opt_name, lr, seed=SEED):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    model = make_model(cfg)
    opt = build_full_optimizer(model, opt_name, lr)
    hist = {"step": [], "val": [], "train": []}
    t0 = time.time(); last = float("nan")
    for step in range(1, cfg["steps"] + 1):
        xb, yb = get_batch(train, cfg["batch_size"], cfg["block_size"], cfg["device"], g)
        opt.zero_grad(set_to_none=True)
        _, loss = model(xb, yb); loss.backward(); opt.step()
        last = loss.item()
        if step == 1 or step % cfg["eval_every"] == 0 or step == cfg["steps"]:
            v = eval_val(model, val, cfg["batch_size"], cfg["block_size"], cfg["device"])
            hist["step"].append(step); hist["val"].append(v); hist["train"].append(last)
    return {"opt": opt_name, "lr": lr, "final_val": hist["val"][-1], "best_val": min(hist["val"]),
            "history": hist, "seconds": time.time()-t0, "model": model}

def train_split(cfg, train, val, b_opt, t_opt, b_lr, t_lr, seed=SEED):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    model = make_model(cfg)
    opt_b, opt_t = build_split_optimizers(model, cfg["split_at"], b_opt, t_opt, b_lr, t_lr)
    hist = {"step": [], "val": [], "train": []}
    t0 = time.time(); last = float("nan")
    for step in range(1, cfg["steps"] + 1):
        xb, yb = get_batch(train, cfg["batch_size"], cfg["block_size"], cfg["device"], g)
        opt_b.zero_grad(set_to_none=True); opt_t.zero_grad(set_to_none=True)
        _, loss = model(xb, yb); loss.backward(); opt_b.step(); opt_t.step()
        last = loss.item()
        if step == 1 or step % cfg["eval_every"] == 0 or step == cfg["steps"]:
            v = eval_val(model, val, cfg["batch_size"], cfg["block_size"], cfg["device"])
            hist["step"].append(step); hist["val"].append(v); hist["train"].append(last)
    return {"bottom_opt": b_opt, "top_opt": t_opt, "bottom_lr": b_lr, "top_lr": t_lr,
            "final_val": hist["val"][-1], "best_val": min(hist["val"]),
            "history": hist, "seconds": time.time()-t0, "model": model}

print("Library ready.")


## C. FLOP meter + train-until-budget


In [ ]:
from collections import defaultdict
from contextlib import contextmanager
from torch.utils._python_dispatch import TorchDispatchMode
from torch.utils.flop_counter import FlopCounterMode


def _numel_from_args(args, kwargs):
    n = 0
    for a in args:
        if torch.is_tensor(a):
            n = max(n, a.numel())
    for a in (kwargs or {}).values():
        if torch.is_tensor(a):
            n = max(n, a.numel())
    return n


def _matmul_flops(a, b):
    if a.ndim < 2 or b.ndim < 2:
        return 0
    M, K = a.shape[-2], a.shape[-1]
    N = b.shape[-1]
    batch = 1
    for d in a.shape[:-2]:
        batch *= d
    return int(batch * 2 * M * N * K)


class PointwiseFlopMode(TorchDispatchMode):
    def __init__(self):
        super().__init__()
        self.flops = 0
        self.by_op = defaultdict(int)

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        out = func(*args, **kwargs)
        raw = getattr(func, "__name__", str(func))
        if hasattr(func, "name"):
            try:
                raw = func.name()
            except Exception:
                pass
        name = raw.split("::")[-1]
        base = name.split(".")[0]
        fl = 0
        tensors = [a for a in args if torch.is_tensor(a)]
        if base in ("mm", "matmul", "bmm"):
            if len(tensors) >= 2:
                fl = _matmul_flops(tensors[0], tensors[1])
        elif base in ("addmm", "baddbmm"):
            if len(tensors) >= 3:
                fl = _matmul_flops(tensors[1], tensors[2])
        elif base in (
            "add", "add_", "sub", "sub_", "mul", "mul_", "div", "div_",
            "pow", "pow_", "neg", "neg_", "sqrt", "rsqrt", "exp", "log",
            "lerp", "lerp_", "clamp", "clamp_", "where",
        ):
            fl = _numel_from_args(args, kwargs)
        elif base in ("addcdiv", "addcdiv_", "addcmul", "addcmul_"):
            fl = 3 * _numel_from_args(args, kwargs)
        if fl:
            self.flops += int(fl)
            self.by_op[base] += int(fl)
        return out


def _as_opt_list(opt):
    if isinstance(opt, list):
        out = []
        for o in opt:
            out.extend(_as_opt_list(o))
        return out
    if hasattr(opt, "optimizers") and isinstance(getattr(opt, "optimizers"), list):
        return list(opt.optimizers)
    return [opt]


def measure_train_step_flops(model, opt, idx, targets):
    if hasattr(opt, "zero_grad") and not isinstance(opt, list):
        opt.zero_grad(set_to_none=True)
        opts_for_step = opt
    else:
        opts = _as_opt_list(opt)
        for o in opts:
            o.zero_grad(set_to_none=True)
        opts_for_step = opts

    model.train()
    with FlopCounterMode(display=False) as fc:
        _, loss = model(idx, targets)
        loss.backward()
    fwd_bwd = int(fc.get_total_flops())

    mode = PointwiseFlopMode()
    with mode:
        if isinstance(opts_for_step, list):
            for o in opts_for_step:
                o.step()
        else:
            opts_for_step.step()
    opt_flops = int(mode.flops)
    total = fwd_bwd + opt_flops
    return {
        "fwd_bwd_flops": fwd_bwd,
        "opt_flops": opt_flops,
        "total_flops": total,
        "opt_share": opt_flops / total if total else 0.0,
        "loss": float(loss.detach()),
        "opt_by_op": dict(sorted(mode.by_op.items(), key=lambda kv: -kv[1])[:12]),
    }


def _build_opt_bundle(model, kind, opt_name=None, lr=None, bottom_opt=None, top_opt=None,
                      bottom_lr=None, top_lr=None, split_at=None):
    if kind == "full":
        opt = build_full_optimizer(model, opt_name, lr)
        return opt, opt
    opt_b, opt_t = build_split_optimizers(
        model, split_at=split_at, bottom_opt=bottom_opt, top_opt=top_opt,
        bottom_lr=bottom_lr, top_lr=top_lr,
    )
    return [opt_b, opt_t], CompositeOptimizer([opt_b, opt_t], "split")


def calibrate_step_flops(model, opt, train, cfg, g, warmup=1, measure=3):
    """Run warmup + measure steps. Return steady-state FLOPs/step and per-step logs."""
    logs = []
    for i in range(warmup + measure):
        xb, yb = get_batch(train, cfg["batch_size"], cfg["block_size"], cfg["device"], g)
        stats = measure_train_step_flops(model, opt, xb, yb)
        stats["calib_i"] = i
        logs.append(stats)
    steady = logs[warmup:]
    per_step = float(np.mean([s["total_flops"] for s in steady]))
    opt_share = float(np.mean([s["opt_share"] for s in steady]))
    return {
        "flops_per_step": per_step,
        "opt_share": opt_share,
        "fwd_bwd": float(np.mean([s["fwd_bwd_flops"] for s in steady])),
        "opt_flops": float(np.mean([s["opt_flops"] for s in steady])),
        "logs": logs,
    }


@torch.no_grad()
def _eval(model, val, cfg):
    return eval_val(model, val, cfg["batch_size"], cfg["block_size"], cfg["device"])


def train_until_flops(
    cfg, train, val, flop_budget, *,
    kind="full",
    opt_name=None, lr=None,
    bottom_opt=None, top_opt=None, bottom_lr=None, top_lr=None,
    seed=SEED,
    calib_warmup=CALIB_WARMUP,
    calib_measure=CALIB_MEASURE,
    label="run",
):
    """
    Train until cumulative FLOPs >= flop_budget.
    Warmup+measure steps are real training and count toward the budget.
    Steady-state FLOPs/step is the mean of the measure window (warmup excluded from that mean only).
    """
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = make_model(cfg)
    opt, opt_for_zero = _build_opt_bundle(
        model, kind, opt_name=opt_name, lr=lr,
        bottom_opt=bottom_opt, top_opt=top_opt,
        bottom_lr=bottom_lr, top_lr=top_lr,
        split_at=cfg["split_at"],
    )

    t0 = time.time()
    calib = calibrate_step_flops(
        model, opt, train, cfg, g, warmup=calib_warmup, measure=calib_measure
    )
    flops_per_step = calib["flops_per_step"]
    # charge calibration steps at their measured totals
    cum = float(sum(s["total_flops"] for s in calib["logs"]))
    steps_done = len(calib["logs"])
    last_train = float(calib["logs"][-1]["loss"])

    hist = {"step": [], "val": [], "train": [], "flops": []}
    def record():
        v = _eval(model, val, cfg)
        hist["step"].append(steps_done)
        hist["val"].append(v)
        hist["train"].append(last_train)
        hist["flops"].append(cum)
        return v

    record()
    eval_every = cfg["eval_every"]
    max_steps = int(flop_budget / max(flops_per_step, 1)) + calib_warmup + calib_measure + 5

    while cum < flop_budget and steps_done < max_steps:
        xb, yb = get_batch(train, cfg["batch_size"], cfg["block_size"], cfg["device"], g)
        if hasattr(opt_for_zero, "zero_grad"):
            opt_for_zero.zero_grad(set_to_none=True)
        else:
            for o in _as_opt_list(opt):
                o.zero_grad(set_to_none=True)
        _, loss = model(xb, yb)
        loss.backward()
        if isinstance(opt, list):
            for o in opt:
                o.step()
        else:
            opt.step()
        last_train = float(loss.item())
        cum += flops_per_step
        steps_done += 1
        if steps_done % eval_every == 0 or cum >= flop_budget:
            record()

    # final eval if last record wasn't at end
    if not hist["step"] or hist["step"][-1] != steps_done:
        record()

    result = {
        "kind": kind,
        "label": label,
        "final_val": hist["val"][-1],
        "best_val": min(hist["val"]),
        "steps": steps_done,
        "flops": cum,
        "flops_per_step": flops_per_step,
        "opt_share": calib["opt_share"],
        "fwd_bwd_per_step": calib["fwd_bwd"],
        "opt_flops_per_step": calib["opt_flops"],
        "history": hist,
        "seconds": time.time() - t0,
        "model": model,
        "calib": {k: calib[k] for k in ("flops_per_step", "opt_share", "fwd_bwd", "opt_flops")},
    }
    if kind == "full":
        result.update({"opt": opt_name, "lr": lr})
    else:
        result.update({
            "bottom_opt": bottom_opt, "top_opt": top_opt,
            "bottom_lr": bottom_lr, "top_lr": top_lr,
        })
    return result

print("FLOP trainer ready.")






## D. Data

FineWeb-Edu `sample-10BT` → local token shard. Set `RESET_DATA=True` if replacing an old smaller shard.


In [ ]:
train, val = ensure_data(CFG)
print(f"Ready — train={len(train):,}  val={len(val):,}")
assert len(train) > CFG["block_size"] + 1
assert len(val) > CFG["block_size"] + 1
if not CFG["force_synthetic"] and len(train) < 1_000_000:
    print("NOTE: train shard < 1M tokens — for the bake-off you usually want RESET_DATA=True "
          "and token_budget=~30M on Colab.")


## E. FLOP budget


In [ ]:
print("=" * 64)
print("BUDGET CALIBRATION")
print("=" * 64)

if FLOP_BUDGET is None:
    torch.manual_seed(SEED)
    g = torch.Generator().manual_seed(SEED)
    probe = make_model(CFG)
    n_params = sum(p.numel() for p in unique_params(probe))
    print(f"actual unique params: {n_params:,} ({n_params/1e6:.2f}M)")
    probe_opt = build_full_optimizer(probe, REF_OPT, LOCKED_LR[REF_OPT])
    cal = calibrate_step_flops(
        probe, probe_opt, train, CFG, g,
        warmup=CALIB_WARMUP, measure=CALIB_MEASURE,
    )
    FLOP_BUDGET = cal["flops_per_step"] * REF_STEPS
    print(f"REF {REF_OPT}: FLOPs/step={cal['flops_per_step']:.4e}  "
          f"opt_share={100*cal['opt_share']:.2f}%")
    print(f"Budget = {cal['flops_per_step']:.4e} × {REF_STEPS} = {FLOP_BUDGET:.4e}")
    del probe, probe_opt
else:
    print(f"Using explicit FLOP_BUDGET={FLOP_BUDGET:.4e}")

print(f"\nFLOP_BUDGET = {FLOP_BUDGET:.6e}")
assert FLOP_BUDGET > 0


## F. Bake-off runs

1. Full baselines once.  
2. Each depth cut × each split pair → same FLOP budget.


In [ ]:
print("=" * 64)
print(f"CONVERGED BAKE-OFF  budget={FLOP_BUDGET:.4e}")
print(f"cuts={len(DEPTH_CUTS)}  pairs={len(SPLIT_COMBOS)}  "
      f"split_trials={len(DEPTH_CUTS)*len(SPLIT_COMBOS)}")
print("=" * 64)

CKPT_PATH = OUT_DIR / "bakeoff_checkpoints.json"
if RESUME and CKPT_PATH.exists():
    ckpt = json.loads(CKPT_PATH.read_text())
    print(f"Resuming from {CKPT_PATH} ({len(ckpt.get('full',[]))} full, "
          f"{len(ckpt.get('split',[]))} split saved)")
else:
    ckpt = {"full": [], "split": [], "flop_budget": FLOP_BUDGET, "config": CFG}

def save_ckpt():
    CKPT_PATH.write_text(json.dumps(ckpt, indent=2))

def full_key(opt):
    return opt

def split_key(frac, bottom_opt, top_opt):
    return f"{frac:.0%}|{bottom_opt}|{top_opt}"

done_full = {r["opt"] for r in ckpt.get("full", [])}
done_split = {split_key(r["depth_frac"], r["bottom_opt"], r["top_opt"]) for r in ckpt.get("split", [])}

full_results = list(ckpt.get("full", []))
best_full = None
if full_results:
    best_full = min(full_results, key=lambda x: x["final_val"])

for opt in FULL_OPTS:
    if full_key(opt) in done_full:
        print(f"\n=== FULL {opt} (skip, cached) ===")
        continue
    lr = LOCKED_LR[opt]
    print(f"\n=== FULL {opt} lr={lr:g} ===", flush=True)
    r = train_until_flops(
        CFG, train, val, FLOP_BUDGET,
        kind="full", opt_name=opt, lr=lr, seed=SEED,
        label=f"full:{opt}",
    )
    r.pop("model", None)
    # drop huge history calib logs if any
    if "calib" in r and isinstance(r["calib"], dict):
        r["calib"] = {k: r["calib"][k] for k in ("flops_per_step", "opt_share", "fwd_bwd", "opt_flops") if k in r["calib"]}
    # keep history for plots
    full_results.append(r)
    ckpt["full"] = full_results
    save_ckpt()
    print(
        f"  → steps={r['steps']}  FLOPs={r['flops']:.4e}  "
        f"opt%={100*r['opt_share']:.1f}  val={r['final_val']:.4f}  ({r['seconds']:.1f}s)  [saved]"
    )

best_full = min(full_results, key=lambda x: x["final_val"])

split_results = list(ckpt.get("split", []))
best_by_depth = {}

for cut in DEPTH_CUTS:
    frac, split_at = cut["frac"], cut["split_at"]
    cfg_d = dict(CFG)
    cfg_d["split_at"] = split_at
    print("\n" + "#" * 64)
    print(f"# DEPTH {frac:.0%}  split_at={split_at}  "
          f"(bottom={split_at}, top={cfg_d['n_layer']-split_at})")
    print("#" * 64)

    for bottom_opt, top_opt in SPLIT_COMBOS:
        key = split_key(frac, bottom_opt, top_opt)
        if key in done_split:
            print(f"\n--- {key} (skip, cached) ---")
            continue
        blr, tlr = LOCKED_LR[bottom_opt], LOCKED_LR[top_opt]
        print(f"\n--- {frac:.0%} | {bottom_opt}|{top_opt} "
              f"lr=({blr:g},{tlr:g}) ---", flush=True)
        r = train_until_flops(
            cfg_d, train, val, FLOP_BUDGET,
            kind="split", bottom_opt=bottom_opt, top_opt=top_opt,
            bottom_lr=blr, top_lr=tlr, seed=SEED,
            label=f"{frac:.0%}:{bottom_opt}|{top_opt}",
        )
        r.pop("model", None)
        if "calib" in r and isinstance(r["calib"], dict):
            r["calib"] = {k: r["calib"][k] for k in ("flops_per_step", "opt_share", "fwd_bwd", "opt_flops") if k in r["calib"]}
        r["depth_frac"] = frac
        r["split_at"] = split_at
        split_results.append(r)
        ckpt["split"] = split_results
        save_ckpt()
        done_split.add(key)
        print(
            f"  → steps={r['steps']}  FLOPs={r['flops']:.4e}  "
            f"opt%={100*r['opt_share']:.1f}  val={r['final_val']:.4f}  ({r['seconds']:.1f}s)  [saved]"
        )

    here = [r for r in split_results if r["split_at"] == split_at]
    best_here = min(here, key=lambda x: x["final_val"])
    best_by_depth[split_at] = best_here
    print(f"\nBEST @ {frac:.0%}: {best_here['bottom_opt']}|{best_here['top_opt']}  "
          f"val={best_here['final_val']:.4f}")

best_split = min(split_results, key=lambda x: x["final_val"])

print("\n" + "=" * 64)
print("FULL ranking:")
for i, r in enumerate(sorted(full_results, key=lambda x: x["final_val"]), 1):
    print(f"  {i}. {r['opt']:8s}  val={r['final_val']:.4f}  steps={r['steps']}  "
          f"opt%={100*r['opt_share']:.1f}")

print("\nBEST split per depth:")
for cut in DEPTH_CUTS:
    b = best_by_depth[cut["split_at"]]
    print(f"  {cut['frac']:.0%} (s={cut['split_at']}): "
          f"{b['bottom_opt']}|{b['top_opt']}  val={b['final_val']:.4f}  steps={b['steps']}")

print(f"\nBEST FULL : {best_full['opt']}  val={best_full['final_val']:.4f}")
print(f"BEST SPLIT: {best_split['depth_frac']:.0%} "
      f"{best_split['bottom_opt']}|{best_split['top_opt']}  "
      f"val={best_split['final_val']:.4f}")
gap = best_full["final_val"] - best_split["final_val"]
print(f"Gap (full - split): {gap:+.4f}")
print(f"Checkpoint: {CKPT_PATH}")


## G. Plots + summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
xs = [c["frac"] * 100 for c in DEPTH_CUTS]
ys = [best_by_depth[c["split_at"]]["final_val"] for c in DEPTH_CUTS]
ax.plot(xs, ys, "o-", lw=2, label="best split @ depth")
ax.axhline(best_full["final_val"], color="k", ls="--",
           label=f"best full ({best_full['opt']})")
for c, y in zip(DEPTH_CUTS, ys):
    b = best_by_depth[c["split_at"]]
    ax.annotate(f"{b['bottom_opt']}|{b['top_opt']}", (c["frac"]*100, y),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_xlabel("bottom depth %")
ax.set_ylabel("val loss @ equal FLOPs")
ax.set_title("Best split vs depth")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
for r in sorted(full_results, key=lambda x: x["final_val"]):
    ax.plot(np.array(r["history"]["flops"]) / 1e12, r["history"]["val"],
            label=f"full:{r['opt']} ({r['final_val']:.3f})")
# top splits overall
for r in sorted(split_results, key=lambda x: x["final_val"])[:6]:
    ax.plot(np.array(r["history"]["flops"]) / 1e12, r["history"]["val"], ls="--",
            label=f"{r['depth_frac']:.0%} {r['bottom_opt']}|{r['top_opt']} ({r['final_val']:.3f})")
ax.axvline(FLOP_BUDGET / 1e12, color="k", ls=":", alpha=0.5)
ax.set_xlabel("Cumulative FLOPs (TFLOP)")
ax.set_ylabel("val loss")
ax.set_title("Val vs FLOPs (full + top splits)")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

fig.tight_layout()
path = OUT_DIR / "converged_bakeoff_overview.png"
fig.savefig(path, dpi=150)
plt.show()
print("saved", path)


In [ ]:
# Heatmap: depth × pair
pair_labels = [f"{b}|{t}" for b, t in SPLIT_COMBOS]
mat = np.full((len(DEPTH_CUTS), len(SPLIT_COMBOS)), np.nan)
for i, cut in enumerate(DEPTH_CUTS):
    for j, (b, t) in enumerate(SPLIT_COMBOS):
        hits = [r for r in split_results
                if r["split_at"] == cut["split_at"]
                and r["bottom_opt"] == b and r["top_opt"] == t]
        if hits:
            mat[i, j] = hits[0]["final_val"]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(mat, cmap="viridis_r", aspect="auto")
ax.set_xticks(range(len(pair_labels)))
ax.set_xticklabels(pair_labels, rotation=30, ha="right")
ax.set_yticks(range(len(DEPTH_CUTS)))
ax.set_yticklabels([f"{c['frac']:.0%} (s={c['split_at']})" for c in DEPTH_CUTS])
ax.set_title("Split val @ equal FLOPs (lower better)")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if not np.isnan(mat[i, j]):
            ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center", color="w", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
path = OUT_DIR / "converged_bakeoff_heatmap.png"
fig.savefig(path, dpi=150)
plt.show()
print("saved", path)


In [ ]:
print("=" * 64)
print("FINAL BAKE-OFF RESULTS")
print("=" * 64)
print(f"FLOP_BUDGET : {FLOP_BUDGET:.6e}  (ref={REF_OPT}×{REF_STEPS})")
print(f"Model       : {CFG['n_layer']}L/{CFG['n_head']}H/{CFG['n_embd']}d")
print(f"Depth cuts  : {[(c['frac'], c['split_at']) for c in DEPTH_CUTS]}")
print(f"Best FULL   : {best_full['opt']}  val={best_full['final_val']:.4f}  "
      f"steps={best_full['steps']}")
print(f"Best SPLIT  : {best_split['depth_frac']:.0%} "
      f"{best_split['bottom_opt']}|{best_split['top_opt']}  "
      f"val={best_split['final_val']:.4f}  steps={best_split['steps']}")
gap = best_full["final_val"] - best_split["final_val"]
print(f"Gap (full - split): {gap:+.4f}")
print(f"Artifacts: {OUT_DIR.resolve()}")
print("=" * 64)

def pack_full(r):
    return {k: r[k] for k in (
        "opt", "lr", "final_val", "best_val", "steps", "flops",
        "flops_per_step", "opt_share", "seconds"
    )}

def pack_split(r):
    return {k: r[k] for k in (
        "depth_frac", "split_at", "bottom_opt", "top_opt", "bottom_lr", "top_lr",
        "final_val", "best_val", "steps", "flops", "flops_per_step", "opt_share", "seconds"
    )}

summary = {
    "mode": MODE,
    "study": "converged_bakeoff",
    "flop_budget": FLOP_BUDGET,
    "ref_opt": REF_OPT,
    "ref_steps": REF_STEPS,
    "config": CFG,
    "locked_lr": LOCKED_LR,
    "depth_cuts": DEPTH_CUTS,
    "full_opts": FULL_OPTS,
    "split_combos": SPLIT_COMBOS,
    "full_ranking": [pack_full(r) for r in sorted(full_results, key=lambda x: x["final_val"])],
    "split_ranking": [pack_split(r) for r in sorted(split_results, key=lambda x: x["final_val"])],
    "best_per_depth": {
        str(cut["split_at"]): pack_split(best_by_depth[cut["split_at"]])
        for cut in DEPTH_CUTS
    },
    "best_full": pack_full(best_full),
    "best_split": pack_split(best_split),
    "gap_full_minus_split": gap,
}
path = OUT_DIR / "converged_bakeoff_summary.json"
path.write_text(json.dumps(summary, indent=2))
print("wrote", path)
print("\nOptional next step: re-run top 2–3 configs with SEED=43,44 for error bars.")

# Full time-series for offline analysis (keep this + bakeoff_checkpoints.json)
histories = {
    "flop_budget": FLOP_BUDGET,
    "config": CFG,
    "locked_lr": LOCKED_LR,
    "full": [
        {
            "opt": r["opt"], "lr": r["lr"], "final_val": r["final_val"],
            "steps": r["steps"], "flops": r["flops"], "opt_share": r["opt_share"],
            "seconds": r["seconds"],
            "history": r.get("history", {}),
        }
        for r in full_results
    ],
    "split": [
        {
            "depth_frac": r["depth_frac"], "split_at": r["split_at"],
            "bottom_opt": r["bottom_opt"], "top_opt": r["top_opt"],
            "bottom_lr": r["bottom_lr"], "top_lr": r["top_lr"],
            "final_val": r["final_val"], "steps": r["steps"], "flops": r["flops"],
            "opt_share": r["opt_share"], "seconds": r["seconds"],
            "history": r.get("history", {}),
        }
        for r in split_results
    ],
}
hist_path = OUT_DIR / "bakeoff_histories.json"
hist_path.write_text(json.dumps(histories))
print("wrote", hist_path, "  (full val/train/flops curves per trial)")


## H. Save to Google Drive

Zips everything under `plots/study` (checkpoints, histories, summaries, PNGs) → `MyDrive/Pretraining/bakeoff_<UTC>.zip`.



In [ ]:
# Zip bake-off artifacts → Google Drive (Colab)
from pathlib import Path
import shutil
from datetime import datetime, timezone

try:
    from google.colab import drive
except ImportError:
    raise SystemExit("Not on Colab — copy plots/study manually.")

drive.mount("/content/drive")

SRC = Path("plots/study")
assert SRC.exists(), f"missing {SRC}"

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
drive_dir = Path("/content/drive/MyDrive/Pretraining")
drive_dir.mkdir(parents=True, exist_ok=True)

# Full study folder zip (includes bakeoff_* + any leftover lr_tune_* if present)
zip_path = Path(shutil.make_archive(f"/tmp/bakeoff_{stamp}", "zip", root_dir=SRC))
dest = drive_dir / zip_path.name
shutil.copy2(zip_path, dest)

# Also copy key files individually (easy to open on Drive)
key_dir = drive_dir / f"bakeoff_{stamp}_files"
key_dir.mkdir(parents=True, exist_ok=True)
KEEP = [
    "bakeoff_checkpoints.json",
    "bakeoff_histories.json",
    "converged_bakeoff_summary.json",
    "converged_bakeoff_overview.png",
    "converged_bakeoff_heatmap.png",
    "locked_lr.json",
    "lr_tune_summary.json",
]
copied = []
for name in KEEP:
    p = SRC / name
    if p.exists():
        shutil.copy2(p, key_dir / name)
        copied.append(name)

print("zip:", dest, f"({dest.stat().st_size/1e6:.2f} MB)")
print("files dir:", key_dir)
print("copied keys:", copied)
print("all in plots/study:", sorted(p.name for p in SRC.iterdir()))
